In [1]:
import pandas as pd
import re
from IPython.display import display, HTML

In [17]:
df = pd.read_csv('AI_bert_subs_seed_27_strict.csv')
df.head()

,filename,program,year,text_clean,ai_related,matched_keywords_all,relevant_section,processed,nouns,adjectives,verbs,topic,probability
0,"2016-11-16-19,00-1",de wereld draait door,2016,"Eelco Bosch van Rosenthal, Dirk Jan Roeleven, ...",yes,"['facebook', 'google', 'twitter', 'algoritme']",...uit Amerika.\nVanochtend geland.\nHet Ameri...,documentaire voetspor undateables seizoen rufu...,documentaire voetspor undateables seizoen rufu...,mooi lang nieuw nepnieuws ander voorafgaand Am...,landen zien maken hebben zeggen willen tegenga...,6,0.102652
1,"2016-02-08-23,04-1",jinek,2016,Uitzending bijwonen? Dat kan. Ga naar jinek.kr...,yes,['drones'],...over.\nEn onze tennisdames verrasten de wer...,tennisdame wereld baan klap sensatie nieuws po...,tennisdame wereld baan klap sensatie nieuws po...,regelrecht ander vorig Nederlands internationa...,verrasten meppen beslissen winnen bekennen opl...,0,0.136302
2,GOEDEMORGEN_N-WON02434179,goedemorgen nederland,2023,en Joodse scholen die hun deuren vandaag dicht...,yes,"['ai', 'kunstmatige intelligentie']",...en de neus van de politie.\nWeet u hoe een ...,neus politie drugslab mens aanraking drug init...,neus politie drugslab mens aanraking drug init...,goed bewuster kunstmatig welmoed hoog laag dir...,weten werken weten denken komen vinden maken z...,19,1.000000
3,GOEDEMORGEN_N-WON02298553,goedemorgen nederland,2022,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,"['twitter', 'kunstmatige intelligentie']",...zegt de Britse minister van buitenlandse za...,minister zaak aanval aanval gemeenschap aanval...,minister zaak aanval aanval gemeenschap aanval...,Brits buitenlands roekeloos serieus internatio...,zeggen zeggen nemen doen nemen doen veroordele...,1,0.099202
4,GOEDEMORGEN_N-WON02108661,goedemorgen nederland,2020,888\nWelkom terug bij Goedemorgen Nederland.\n...,yes,['drone'],...er aardig uit.\nIs het een quarantainecoup?...,quarantainecoup filmp laag respect kappersvak ...,quarantainecoup filmp laag respect kappersvak ...,aardig diep moeilijk heel vast flink droog fli...,gekeken proberen opkrijgen doen nemen willen b...,5,1.000000


In [18]:
df.shape

(501, 13)

In [31]:
# -----------------------------
# SETTINGS
# -----------------------------
topic_id = 1    # choose topic
n_examples = 12   # number of rows to display
text_col = "relevant_section"  # or "text_full", etc.

# -----------------------------
# YOUR KEYWORDS
# -----------------------------
keywords = (
    "kunstmatige intelligentie|artificial intelligence|artificiële intelligentie|AI|generatieve AI|"
    "generatieve kunstmatige intelligentie|generatieve artificiële intelligentie|"
    "machine learning|machinaal leren|diep leren|deep learning|neurale netwerken|"
    "large language model|grote taalmodel*|LLM|chatbot*|GPT|ChatGPT|Bard|Claude|"
    "Gemini|LLaMA|openai|kunstmatige intelligentie systeem*|intelligente algoritme*|"
    "slimme algoritme*|automatische besluitvorming|automatisch beslissysteem|"
    "algoritmische besluitvorming|algoritme*|cognitieve technologie*|AI-technologie*|"
    "AI-systeem*|AI-toepassing*|AI-model*|spraakherkenning|beeldherkenning|"
    "computer vision|natuurlijke taalverwerking|natural language processing|NLP|robot|drones|drone"
)

_COMPANY_NAMES = [
    "NVIDIA", "Apple", "Microsoft", "Google", "Alphabet",
    "Meta Platforms", "Facebook", "Tesla", "Oracle",
    "Palantir", "IBM", "Adobe", "Cambricon Technologies",
    "CoreWeave", "Fermi Inc", "Dynatrace", "Tempus AI",
    "SenseTime", "Mobileye", "Aurora Innovation", "UiPath",
    "SoundHound AI", "ASML", "NXP Semiconductors",
    "BE Semiconductor Industries", "ASM International",
    "Adyen", "Just Eat Takeaway", "Booking.com", "Mollie",
    "Picnic", "TomTom", "Swapfiets", "TKH Group",
    "Ordina", "Nedap", "CM.com", "ICT Group",
    "Neways Electronics", "Ctac", "Photon Energy",
    "Almunda Professionals", "Samsung", "Huawei",
    "Sony", "LG", "Baidu", "Tencent",
    "Alibaba", "Douyin", "Cloudflare",
    "Snowflake", "Docker", "Red Hat",
    "Uber", "Bolt", "Grab", "Epic Games",
    "Unity", "Discord"
]

# -----------------------------
# BUILD REGEX
# -----------------------------

def wildcard_to_regex(pattern):
    return pattern.replace("*", r"\w*")

keyword_patterns = [wildcard_to_regex(k) for k in keywords.split("|")]
company_patterns = [re.escape(c) for c in _COMPANY_NAMES]

all_patterns = keyword_patterns + company_patterns

regex = re.compile(r"\b(" + "|".join(all_patterns) + r")\b", re.IGNORECASE)

# -----------------------------
# HIGHLIGHT FUNCTION
# -----------------------------

def highlight_text(text):
    if pd.isna(text):
        return ""
    
    def repl(match):
        return f"<span style='color:red; font-weight:bold; font-size:120%;'>{match.group(0)}</span>"
    
    return regex.sub(repl, text)

# -----------------------------
# FILTER + SAMPLE
# -----------------------------

df_topic = df[df["topic"] == topic_id].copy()
df_sample = df_topic.sample(n=min(n_examples, len(df_topic)), random_state=42)

# -----------------------------
# DISPLAY
# -----------------------------

for i, row in df_sample.iterrows():
    title = row.get("title", "")
    text = row.get(text_col, "")

    # also show filename, program and year  
    filename = row.get("filename", "")
    program = row.get("program", "")
    year = row.get("year", "")
    meta_info = f"<small><em>{filename} | {program} | {year}</em></small>"
    
    
    html = f"""
    <div style="margin-bottom:30px;">
        {meta_info}
        <h4>{highlight_text(title)}</h4>
        <p>{highlight_text(text)}</p>
    </div>
    """
    
    display(HTML(html))

In [ ]:

# optional: save to Excel
# df_topic.to_excel(f"topic_{topic_id}_subset.xlsx", index=False)